# Haruka v2: the default model and how every figure was fitted

Every figure in the paper uses one model configuration: the **composition-gated shared-decoder Haruka**
(`bench_kras_xenium/run_haruka_separate_batch_heads.py`, which wraps `bench_mouse_brain_scviva/run_haruka_shared_decoder.py`).

* **z (background)**: per-cell latent shared by control and condition cells.
* **s (salient)**: condition-specific latent. The decoder sets s = 0 for control cells, so control s is never used
  and every z-vs-s readout is computed on condition-arm cells only.
* **Decoders**: a negative-binomial expression decoder plus two niche heads, the cell-type composition of the k = 20
  spatial neighbours and the composition-gated neighbour mean expression (`spatial_weight 100`).
* **Salient composition shift**: D([z, s]) − D([z, 0]), what s adds to the predicted neighbourhood.
* **z/s independence**: one-way ridge penalty (z ← s, weight 10).
* **Batch**: injected into the decoders only (`decoder_batch_dim 8`); separate batch projectors for the
  composition and neighbour-mean heads. z and s encoders are batch-blind.
* **Training**: 150 epochs, 32-d latents, sparse augmentation.

In [1]:
import sys; sys.path.insert(0, '/home/yancui/haruka_v2/notebooks')
from repro import run, require, show, STAGES, H, PY
# Default: figure stage only (redraw from saved analysis outputs). Turn on earlier stages to rerun them.
# STAGES.update(data=True, fit=True, analysis=True)
print(STAGES)

{'data': False, 'fit': False, 'analysis': False, 'figure': True}


## Default arguments (shared by all fits)

In [2]:
DEFAULT = """--seed 0 --max_epochs 150 \
  --n_latent 32 --n_hidden 256 --n_layers 1 --decoder_layers 2 --k_niche 20 --neighbor_embedding_dim 16 \
  --composition_loss_weight 1 --neighbor_mean_loss_weight 1 --neighbor_expression_mode composition_gated --spatial_weight 100 \
  --background_wasserstein_penalty 0 --zs_independence_penalty 10 --decoder_batch_dim 8 --batch_injection_mode decoder_only \
  --augmentation_mode sparse --num_workers 4 --direct_only"""
print(DEFAULT)

--seed 0 --max_epochs 150   --n_latent 32 --n_hidden 256 --n_layers 1 --decoder_layers 2 --k_niche 20 --neighbor_embedding_dim 16   --composition_loss_weight 1 --neighbor_mean_loss_weight 1 --neighbor_expression_mode composition_gated --spatial_weight 100   --background_wasserstein_penalty 0 --zs_independence_penalty 10 --decoder_batch_dim 8 --batch_injection_mode decoder_only   --augmentation_mode sparse --num_workers 4 --direct_only


## Per-dataset arguments

| figure | data | batch key | condition (target) | cell type key | likelihood / HVG | extra |
|---|---|---|---|---|---|---|
| Fig. 2 benchmark | paired mouse-brain simulation (5 section pairs) | slice | disease | cell type | NB | λ sweep, λ = 10 shown |
| Fig. 3 AD plaque | STARmap AD, CODEX CNS | slice / donor | disease | cell type | Normal, pretransformed | |
| Exercise | old control vs old exercise MERFISH | run | exercise | cell type | NB | |
| PEtracer | M2-1 lineage section | graph | malignant cells in tumours | cell type | NB | clone never seen |
| Lung fibrosis | GSE250346 Xenium, 1.63 M cells, 343 genes | core (`sample`) | fibrosis | cell_type | NB, all genes | |
| KRAS inhibitor | Xenium 5K, 920 k cells | slice | KRASi_treated | celltype | NB, 2000 HVG | 3 shuffled-s nulls |

The exact command for each is in that figure's notebook.

## Batch runner used for the real-data fits

In [3]:
print(open(H / 'bench_real_discovery/run_all_compgated_samplebatch_sepheads.sh').read())

#!/usr/bin/env bash
set -euo pipefail

ROOT=/home/yancui/haruka_v2
PY=/home/yancui/.conda/envs/haruka2/bin/python
RUNNER=$ROOT/bench_kras_xenium/run_haruka_separate_batch_heads.py
CC=$ROOT/bench_kras_xenium/rebuild_condition_only_salient_cc.py
PLOT=$ROOT/bench_real_discovery/plot_sample_batch_umaps.py
OUT=${HARUKA_RESULTS_ROOT:-$ROOT/results_real_compgated_nb_final}
GROUP=${1:?usage: run_all_compgated_samplebatch_sepheads.sh GROUP}

common=(
  --out_root "$OUT" --seed 0 --max_epochs 150
  --n_latent 32 --n_hidden 256 --n_layers 1 --decoder_layers 2
  --k_niche 20 --neighbor_embedding_dim 16
  --composition_loss_weight 1 --neighbor_mean_loss_weight 1
  --neighbor_expression_mode composition_gated --spatial_weight 100
  --background_wasserstein_penalty 0 --zs_independence_penalty 10
  --decoder_batch_dim 8 --batch_injection_mode decoder_only
  --augmentation_mode sparse --num_workers 4 --direct_only
)

run_one() {
  local study=$1 data=$2 batch_size=$3 sample_key=$4 batch_key=$5
  local 

## Outputs of a fit (`<out_root>/<data stem>/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0/`)

`background.npy` (z), `salient.npy` (s), `obs_names.npy`, `background_composition.npy` D([z,0]),
`condition_composition.npy` D([z,s]), `salient_composition_shift.npy`, `meta.json` (all arguments + timings),
optionally `model/` and `null_shuffled_s_composition_shift_{k}.npy`.

In [4]:
import json
m = json.load(open(H / 'results_kras_te_null/mouselung_kras_xenium_clean_noC3/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0/meta.json'))
{k: m[k] for k in list(m)[:40] if not isinstance(m[k], (list, dict))}

{'method': 'haruka_symmetric_full_niche',
 'seed': 0,
 'runtime_s': 2656.9518597126007,
 'max_epochs': 150,
 'batch_size': 32768,
 'n_latent': 32,
 'n_hidden': 256,
 'n_layers': 1,
 'decoder_layers': 2,
 'k_niche': 20,
 'neighbor_embedding_dim': 16,
 'composition_loss_weight': 1.0,
 'neighbor_mean_loss_weight': 1.0,
 'neighbor_expression_mode': 'composition_gated',
 'spatial_weight': 100.0,
 'background_wasserstein_penalty': 0.0,
 'zs_independence_penalty': 10.0,
 'symmetric_zs_independence': False,
 'detach_target_z_for_composition': False,
 'mask_salient_mean_on_background': False,
 'mender_scale': 6,
 'mender_radius': 15.0,
 'mender_neighbors': 15,
 'cell_type_key': 'celltype',
 'batch_key': 'slice_id',
 'batch_injection_mode': 'decoder_only',
 'assay_type': 'sequencing',
 'assay_reconstruction': 'sequencing nb likelihood',
 'decoder_batch_injection': True,
 'decoder_batch_embedding_dim': 8,
 'neighbor_input_mode': 'normalize',
 'augmentation_mode': 'sparse',
 'neighbor_target_imple

## Figure